In [3]:
from pymongo import MongoClient
import certifi
from dotenv import load_dotenv
import os

load_dotenv()

uri = os.getenv("MONGO_URI")

client = MongoClient(
    uri,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=30000,
)

print(client.admin.command("ping"))

{'ok': 1}


In [4]:
from configs.config import Config
from src.mongo_utils import get_collection

collection_raw = get_collection(
    uri=Config.mongo.URI,
    db_name=Config.mongo.DB_NAME,
    collection_name=Config.mongo.COLLECTION_RAW,
)

print(collection_raw)

ModuleNotFoundError: No module named 'configs'

In [5]:
import sys
import os
from pathlib import Path

sys.path.insert(0, os.path.abspath(".."))

from dotenv import load_dotenv

PROJECT_ROOT = Path(os.path.abspath(".."))

load_dotenv(PROJECT_ROOT / ".env")

from configs.config import Config
from src.mongo_utils import (
    get_collection,
    find_all_as_dicts,
)

Config.paths.ensure_dirs()

print("[OK] Environment berhasil dimuat.")

[OK] Environment berhasil dimuat.


In [6]:
collection_raw = get_collection(
    uri=Config.mongo.URI,
    db_name=Config.mongo.DB_NAME,
    collection_name=Config.mongo.COLLECTION_RAW,
)

print("[OK] Collection raw_comments berhasil dimuat.")

2026-05-19 22:58:15 | INFO | mongo_utils | Koneksi MongoDB berhasil dibuat.
[OK] Collection raw_comments berhasil dimuat.


In [7]:
projection = {
    "_id": 0,
    "comment_id": 1,
    "video_id": 1,
    "text": 1,
    "text_original": 1,
    "published_at": 1,
    "like_count": 1,
    "source_title": 1,
    "source_url": 1,
}

raw_records = find_all_as_dicts(
    collection_raw,
    projection=projection,
)

print(f"Total raw comments ditemukan: {len(raw_records):,}")

if len(raw_records) > 0:
    print("\nContoh dokumen:")
    print(raw_records[0])

Total raw comments ditemukan: 14,107

Contoh dokumen:
{'comment_id': 'Ugz9rfTI7kkUQZNgNft4AaABAg', 'video_id': '_8H_-uNHKL4', 'text': 'Aiman perlu mengundang rektor yang ini yang setuju like', 'text_original': 'Aiman perlu mengundang rektor yang ini yang setuju like', 'like_count': 3212, 'published_at': '2025-08-22T10:17:21Z', 'source_url': 'https://www.youtube.com/watch?v=_8H_-uNHKL4&t=351s', 'source_title': '#UGMMENJAWAB IJAZAH JOKO WIDODO'}


In [8]:
if not raw_records:
    raise ValueError("Collection raw_comments kosong.")

required_fields = [
    "comment_id",
    "video_id",
    "text_original",
]

sample_keys = raw_records[0].keys()

missing_fields = [
    field
    for field in required_fields
    if field not in sample_keys
]

if missing_fields:
    raise ValueError(
        f"Field berikut tidak ditemukan: {missing_fields}"
    )

print("[OK] Schema MongoDB valid.")

[OK] Schema MongoDB valid.


In [9]:
def normalize_text(text):
    if text is None:
        return ""

    text = str(text)

    text = text.replace("\u200b", " ")
    text = text.replace("\xa0", " ")

    text = " ".join(text.split())

    return text.strip()


label_rows = []

for doc in raw_records:

    text_original = normalize_text(
        doc.get("text_original") or doc.get("text") or ""
    )

    # Skip kosong
    if not text_original:
        continue

    # Skip hanya simbol
    if not any(ch.isalnum() for ch in text_original):
        continue

    # Skip terlalu pendek
    if len(text_original.split()) < 3:
        continue

    label_rows.append({
        "comment_id": doc.get("comment_id", ""),
        "video_id": doc.get("video_id", ""),
        "source_title": doc.get("source_title", ""),
        "source_url": doc.get("source_url", ""),
        "published_at": doc.get("published_at", ""),
        "like_count": doc.get("like_count", 0),

        # LABEL STUDIO FIELD
        "text": text_original,
    })

print(f"Total data setelah filtering: {len(label_rows):,}")

Total data setelah filtering: 13,493


In [10]:
import pandas as pd

labeling_df = pd.DataFrame(label_rows)

if labeling_df.empty:
    raise ValueError(
        "Tidak ada data yang lolos filtering."
    )

# Hapus duplicate comment_id
labeling_df = labeling_df.drop_duplicates(
    subset=["comment_id"]
)

labeling_df = labeling_df.reset_index(drop=True)

print(f"Total data setelah deduplikasi: {len(labeling_df):,}")

labeling_df.head(10)

Total data setelah deduplikasi: 13,493


,comment_id,video_id,source_title,source_url,published_at,like_count,text
0,Ugz9rfTI7kkUQZNgNft4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-22T10:17:21Z,3212,Aiman perlu mengundang rektor yang ini yang se...
1,Ugy5ylL5VV1YUmasO2d4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-24T04:30:59Z,175,KEJUJURAN PASTI SEDERHANA. KEBOHONGAN PASTI BE...
2,UgxSvXyORutffHhhhP14AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-22T09:01:41Z,314,Dari wajah ketiga ini sangat tegang dlm berarg...
3,UgyWLKMfSxU2GetlTXF4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-22T11:52:03Z,194,"Keterangannya ketahuan diseting,dah gak bisa b..."
4,UgyGJYZSgnhooLocR-B4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2026-02-11T13:25:51Z,1,"Ya UGM juga harus Tegas dan bisa menjawab, kea..."
5,UgytnVjgbrmt06dnVA94AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2026-01-28T05:43:43Z,0,"Jng dipengaruhi oleh Kekuasaan yg ada, teruska..."
6,UgxkxqrAgMipvCAa6iZ4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-23T11:03:11Z,142,Ini mah cuma konten bukan pernyataan resmi did...
7,UgwilY1NhsRn4TNdbEd4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-22T11:38:00Z,58,Pengurus ugm tanggung jawab bukan hanya di dun...
8,UgyuYuBEzYQSaGH3lX94AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-22T09:50:36Z,119,MASALAH PAS KASMOJO AJA DARI PEMBIMBING SKRIPS...
9,UgxVLm7CDCZCr-yqzTJ4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2025-08-22T09:51:48Z,533,SE level petinggi UGM Tidak Bisa melihat Photo...


In [11]:
TARGET_LABELING_COUNT = 10000
RANDOM_SEED = 42

if len(labeling_df) > TARGET_LABELING_COUNT:

    export_df = labeling_df.sample(
        n=TARGET_LABELING_COUNT,
        random_state=RANDOM_SEED,
    )

else:
    export_df = labeling_df.copy()

export_df = export_df.reset_index(drop=True)

print(f"Total data export: {len(export_df):,}")

Total data export: 10,000


In [12]:
export_df["label"] = None
export_df["annotator"] = None
export_df["notes"] = None

export_df.head(10)

,comment_id,video_id,source_title,source_url,published_at,like_count,text,label,annotator,notes
0,Ugyl0rpwQyayVtB1Wix4AaABAg,g7Et92hQ_yE,"🔴LIVE Scan Ijazah Jokowi Resmi Ditunjukan, Asl...",https://www.youtube.com/watch?v=g7Et92hQ_yE&t=...,2025-11-26T09:55:59Z,0,Ijazah hilang bisa minta salinan dari UGM,None,None,None
1,UgyMY2rWPOczvFnRgfN4AaABAg,_8H_-uNHKL4,#UGMMENJAWAB IJAZAH JOKO WIDODO,https://www.youtube.com/watch?v=_8H_-uNHKL4&t=...,2026-02-24T10:58:23Z,0,Semakin hari semakin nyata sampai tgl 24 feb 2...,None,None,None
2,Ugx6_x2RI9MMlGvmvft4AaABAg,UEocTRWhtsE,"[FULL] AKHIRNYA MELIHAT IJAZAH JOKOWI, ROY SUR...",https://www.youtube.com/watch?v=UEocTRWhtsE,2025-12-17T01:16:35Z,0,Yang menentukan keaslian ijasah pk jokowi buka...,None,None,None
3,Ugy5oK5SyIMQTbut3_V4AaABAg,g7Et92hQ_yE,"🔴LIVE Scan Ijazah Jokowi Resmi Ditunjukan, Asl...",https://www.youtube.com/watch?v=g7Et92hQ_yE&t=...,2025-12-09T15:35:43Z,0,"Chanel aneh, yg ngompor2in rakyat, bikin peran...",None,None,None
4,Ugwta-RmGAQ9WLCw6FF4AaABAg,UE2gk8vMgBQ,Gebrakan Baru Rocky Gerung Memasuki Drama Kasu...,https://www.youtube.com/watch?v=UE2gk8vMgBQ,2026-01-29T13:40:01Z,0,kasian si Damai kelihatan lelah dn stres krn k...,None,None,None
5,UgywDrSi0BeY5jmNd6R4AaABAg,LgaDFmaTpak,"[FULL] Babak Baru! Kasus Ijazah Jokowi, Roy Cs...",https://www.youtube.com/watch?v=LgaDFmaTpak,2026-01-17T03:57:02Z,52,"Andi aswan keliatan bohongnya, jilatannya kuat...",None,None,None
6,Ugy0gYdy0AXRdNusmNV4AaABAg,TcNSqpBPle4,"FULL Dihina Soal Ijazah, Jokowi Akhirnya Angka...",https://www.youtube.com/watch?v=TcNSqpBPle4,2026-03-20T15:08:44Z,0,Cepat / lambat menutupi kebohongan sama saja m...,None,None,None
7,UgzW6E5xFEstN1J0WyZ4AaABAg,tQKnVXSL7nc,[FULL] Perdana! Sidang MK Uji Materi Gugatan R...,https://www.youtube.com/watch?v=tQKnVXSL7nc,2026-02-10T14:00:43Z,20,Menurut pandangan sya gk bakal sanggup untuk d...,None,None,None
8,UgzrdCU3BIgOnTGJvIB4AaABAg,HcwZx39bP80,"Kasus Ijazah Jokowi, Apa yang Sebenarnya Terja...",https://www.youtube.com/watch?v=HcwZx39bP80,2025-12-18T09:14:35Z,0,Semua kasus hukum yang berkaitan dengan peting...,None,None,None
9,UgycsYlwWyb0j4BJj5t4AaABAg,tQKnVXSL7nc,[FULL] Perdana! Sidang MK Uji Materi Gugatan R...,https://www.youtube.com/watch?v=tQKnVXSL7nc,2026-02-10T13:42:56Z,0,"sudah di bilang, tempat yg jitu utk di datangi...",None,None,None


In [13]:
print("Jumlah komentar per video:\n")

print(
    export_df["video_id"]
    .value_counts()
    .head(20)
)

print("\nJumlah total final:")
print(len(export_df))

Jumlah komentar per video:

video_id
TcNSqpBPle4    899
_8H_-uNHKL4    804
UEocTRWhtsE    791
5btOJ_Wru2k    768
g7Et92hQ_yE    763
tQKnVXSL7nc    761
uxDGtbE72jk    733
LgaDFmaTpak    714
HcwZx39bP80    674
BMcbB3SlU1M    664
4yjXBLcjynU    660
u4zZ7gHRBaU    618
UE2gk8vMgBQ    613
UTvTHgjI_QI    538
Name: count, dtype: int64

Jumlah total final:
10000


In [14]:
import json

labeling_dir = Config.paths.OUTPUTS / "labeling"

labeling_dir.mkdir(
    parents=True,
    exist_ok=True,
)

json_path = labeling_dir / "label_studio_dataset.json"

records = export_df.to_dict(
    orient="records"
)

with open(
    json_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        records,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"[OK] JSON berhasil disimpan:")
print(json_path)

[OK] JSON berhasil disimpan:
C:\jokowi_sentiment_project\outputs\labeling\label_studio_dataset.json


In [15]:
csv_path = labeling_dir / "labeling_dataset.csv"

export_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"[OK] CSV berhasil disimpan:")
print(csv_path)

[OK] CSV berhasil disimpan:
C:\jokowi_sentiment_project\outputs\labeling\labeling_dataset.csv


In [16]:
print("Preview dataset labeling:\n")

export_df[
    [
        "comment_id",
        "video_id",
        "text",
        "like_count",
    ]
].head(10)

Preview dataset labeling:



,comment_id,video_id,text,like_count
0,Ugyl0rpwQyayVtB1Wix4AaABAg,g7Et92hQ_yE,Ijazah hilang bisa minta salinan dari UGM,0
1,UgyMY2rWPOczvFnRgfN4AaABAg,_8H_-uNHKL4,Semakin hari semakin nyata sampai tgl 24 feb 2...,0
2,Ugx6_x2RI9MMlGvmvft4AaABAg,UEocTRWhtsE,Yang menentukan keaslian ijasah pk jokowi buka...,0
3,Ugy5oK5SyIMQTbut3_V4AaABAg,g7Et92hQ_yE,"Chanel aneh, yg ngompor2in rakyat, bikin peran...",0
4,Ugwta-RmGAQ9WLCw6FF4AaABAg,UE2gk8vMgBQ,kasian si Damai kelihatan lelah dn stres krn k...,0
5,UgywDrSi0BeY5jmNd6R4AaABAg,LgaDFmaTpak,"Andi aswan keliatan bohongnya, jilatannya kuat...",52
6,Ugy0gYdy0AXRdNusmNV4AaABAg,TcNSqpBPle4,Cepat / lambat menutupi kebohongan sama saja m...,0
7,UgzW6E5xFEstN1J0WyZ4AaABAg,tQKnVXSL7nc,Menurut pandangan sya gk bakal sanggup untuk d...,20
8,UgzrdCU3BIgOnTGJvIB4AaABAg,HcwZx39bP80,Semua kasus hukum yang berkaitan dengan peting...,0
9,UgycsYlwWyb0j4BJj5t4AaABAg,tQKnVXSL7nc,"sudah di bilang, tempat yg jitu utk di datangi...",0
